In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
os.chdir("..")
home = os.getcwd()
print(home)

c:\Users\HishamMo\Desktop\999_cs\animal-stress-reducer


In [4]:
meta_song_path = os.path.join(home,"metadata_song.csv")
df = pd.read_csv(meta_song_path)
df.head()

,id,english_name,country,lat,lon,quality,length,time,date,sample_rate,remarks,sex,animal_seen,temp,label
0,1093386,Great Tit,China,39.9915,116.3113,A,0:35,6:09,2026-04-04,44100,NaN,NaN,yes,NaN,song
1,1092854,Great Tit,Cyprus,34.7618,32.5144,A,0:31,17:00,2026-03-22,24000,NaN,male,no,NaN,song
2,1092853,Great Tit,Cyprus,34.7618,32.5144,A,0:14,17:00,2026-03-22,24000,NaN,male,no,NaN,song
3,1092510,Great Tit,France,46.6716,4.8614,A,0:19,10:00,2026-02-20,24000,L'enregistrement n'a pas été modifié. Enregist...,NaN,no,NaN,song
4,1092503,Great Tit,Italy,39.6118,16.7489,A,0:49,15:43,2026-03-29,44100,NaN,male,yes,NaN,song


In [ ]:
remarks_mapping = {
    # --- TYPICAL (General, Technical, SNR, or Empty) ---
    None: "typical", "nan": "typical", " ": "typical", "": "typical",
    "SNR 52dB(ISO/ITU)": "typical", "51 dB (ISO/ITU). ": "typical",
    "SNR 54db, (ISO/ITU)": "typical", "hpf": "typical", "HPF": "typical",
    "SNR 57 dB (ISO / ITU)": "typical", "SNR 56 dB (ISO/ITU)": "typical",
    "HPF 1000Hz -6dB  [SNR 55dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "[SNR 60dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "[SNR 57dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    " [SNR 57dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "[SNR 63dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "[SNR 54dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "[SNR 67dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "[SNR 53dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "[SNR 56dB(ISO/ITU)]": "typical", "[SNR 52dB(ISO/ITU)]": "typical",
    "[SNR 58dB(ISO/ITU)]": "typical", " [SNR 50dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    " [SNR 55dB(ISO/ITU)](https://xeno-canto.org/article/275)": "typical",
    "SNR 44db, (ISO/ITU).": "typical", "SNR 40dB(ISO/ITU)    Habitat: Deciduous spinney in rural village, backing onto open farmland  ": "typical",
    "High-Pass-filtered with frequency 300 Hz and roll-off 6 dB. SNR-Quality: C (21 dB ISO / ITU)  ": "typical",
    "High-Pass-filtered with frequency 300 Hz and roll-off 6 dB": "typical",
    "hpf, nr, intervals shortened": "typical", "none modified or filtered.": "typical",
    "high pass filter, noise reduction": "typical", "normalized -3 dB, HPF": "typical",
    "Turkestan Tit'": "typical", "the very second he left the nestbox he started singing en flew off": "typical",
    "Check my website birdsongs.it for more details": "typical", "Check my personal website: www.birdsongs.it": "typical",
    "Loud and Clear": "typical", "Cleaned": "typical", "Telinga": "typical", "PS12": "typical", "PS24": "typical",
    "Resuming adult. ": "typical", "rainy and cold day": "typical", "3em ind": "typical",
    "Thirst common type of song in the year (called spring song) 3 different males": "typical",
    "Notice that the first note of each phrase is much higher pitched and longer than the second note.": "typical",
    "lonely bird, same as XC122926, one from most common motives": "typical",
    "The bird was in bad condition. The plumage was a mess.": "typical",

    # --- DUET ---
    "Two birds in duet. See here <https://ebird.org/checklist/S129433248>": "duet",
    "[SNR 56dB(ISO/ITU)](https://xeno-canto.org/article/275) Two birds singing": "duet",
    "Two different singers, the closest one was for sure a male": "duet",
    "interacting pair": "duet",
    "Isolated two males - in the background XC532299  ": "duet",
    "Great Tit song phrases seemed to be synchronised with those of a nearby Song Thrush, so that both sang in the gaps between the other's song phrases. ": "duet",
    "SNR 52 dB (ISO/ITU). Two males singing the same phrase": "duet",

    # --- LOCAL ---
    "Local dialect": "local",
    "Local dialect, same bird as: XC150231": "local",
    "Local dialect, same bird as: XC150231 and XC150232": "local",
    "Interesting dialect over some  adjacent pairs": "local",
    "interesting pre-verb and arrhythmic syllable - next bird by the XC175102 - some kind local micro dialect (at least 2 more birds around)": "local",

    # --- STRANGE ---
    "Strange song": "strange",
    "Strange song. Known to be Great Tit from sequence before and after.": "strange",
    "Unusual songtype": "strange",
    "Song variant, taken during overnight recording session. Somewhat overmodulated :-( ": "strange",
    "aberrant song, maybe imitating alarm call of Turdus merula": "strange",
    "song with  very fast two-note strophes": "strange",
    "song of a positively identified male changing strophe type in the middle of the recording from a two-note song to a three-note song": "strange",
    "Part of longer song composed from that unusual motive because of chaining the  syllables throughout the course of 2 or 3 element song.": "strange",
    "Interesting motive, on Pair of Marsh Tit (Poecile palustris)": "strange",
    "Was switching between this song and a more normal type.": "strange",

    # --- HOSTILE ---
    "Hostile song with mimicry of alarm call of  this bird  Marsh Tit: XC129207 same as XC129213": "hostile",
    "Song (antaginistic to Singing Blue Tit)": "hostile",
    "Highpass filter: 3.3 kHz    Male great tit singing high up in birch canopy. Song was quickly interupted by an other great tit flying overhead. The recorded tit flew after it and engaged in fight. This is not recorded, the after math of this fight resulted in the tit calling from the same perch it sang from minutes before.": "hostile",

    # --- MIMICRY ---
    "Mimicry of Long-tailed Tit": "mimicry",

    # --- JUVENILE ---
    "juvenile still practicing the main song, in which it fell into the end? No significant modification; habitat: city backyard; see also iNat obs https://www.inaturalist.org/observations/316576409": "juvenile",

    # --- URBAN ---
    "Urban area.": "urban", "Urban area. ": "urban",
    "L'enregistrement n'a pas été modifié. Enregistreur posé dans un jardin d'une maison. ": "urban",
    "Same birds XC169869 & XC169870 & XC169871.  Street noise. Habitat: garden in the suburbs. ": "urban",
    "habitat: city backyard": "urban",
    "Grabado en urbanización de baja densidad de población con abundante arbolado. Grabación amplificada": "urban",
    "in my garden. Song type changes while recording.": "urban",
    "Song from a bird in a tree in a garden.": "urban",

    # --- FOREST/TREES ---
    "Male singing very close to us from a willow tree.": "forest/trees",
    "Great tit singing, perched on an oak branch.": "forest/trees",
    "In dense forest.": "forest/trees",
    "Recording in dense old forest. ": "forest/trees",
    "Habitat: coniferous forest.": "forest/trees",
    "Same bird XC168434. Habitat: coniferous forest.": "forest/trees",
    "Habitat: wet deciduous forest.": "forest/trees",
    "At the rim of a forest.": "forest/trees",
    "Not modified. Mature deciduous forest. ": "forest/trees",
    "Song from a bird in the top of a pine.": "forest/trees",
    "In oak trees in an open area.": "forest/trees",
    "boreal forest": "forest/trees",
    "The recording was filtered in Audacity (high-pass filter [2,500Hz -24 dB]; normalized volume peak amplitude [-3.0 dB]). The habitat is old grown boreal forest.": "forest/trees",
    "Song from a bird in a small tree at the edge of a field.": "forest/trees",
    "Song from a bird moving in a low tree.": "forest/trees",
    "Song from a bird perched in a low tree in the pasture.": "forest/trees",
    "Bird singing from top of a small tree on forest edge bordering extensive farmed vineyards.": "forest/trees",
    "Bird singing from top of small willow clump (Salix sp.).": "forest/trees",
    "Unmodified. Deciduous trees near the village. ": "forest/trees",
    "Unmodified. Deciduous forest edge. ": "forest/trees",
    "a mature hornbeam-beech forest, +9 degrees Celsius, light wind, cloudy, high-passed": "forest/trees",
    "Bird located in forest at some distance. Noise filtered and amplified in Audacity. ": "forest/trees"
}
ppppppdf['remarks'] = df['remarks'].map(remarks_mapping).fillna("typical")

In [7]:
v = df['remarks'].unique()
print(v)

<StringArray>
[     'typical',        'urban', 'forest/trees',         'duet',
      'strange',      'mimicry',        'local',      'hostile',
     'juvenile']
Length: 9, dtype: str


In [9]:
df.head()
df.to_csv(meta_song_path)

In [16]:
meta_alarm_path = os.path.join(home,"metadata.csv")
df2 = pd.read_csv(meta_alarm_path)
df2.head()

,id,english_name,country,lat,lon,quality,length,time,date,sample_rate,remarks,sex,animal_seen,temp,label
0,1088904,Great Tit,Sweden,56.1820,14.8897,A,0:24,10:00,2026-03-15,44100,NaN,male,yes,NaN,alarm_call
1,985820,Great Tit,Netherlands,53.0228,5.9170,A,0:48,08:30,2025-04-04,96000,Highpass filter: 3.3 kHz Male great tit cal...,male,yes,NaN,alarm_call
2,983069,Great Tit,Netherlands,52.9638,6.1637,A,0:40,08:00,2025-03-26,48000,"Edited: -12 dB High-pass filter: 1200 Hz, 6 d...",uncertain,yes,NaN,alarm_call
3,956913,Great Tit,Estonia,59.3601,24.1552,A,0:07,8:00,2024-12-18,32000,usual call of female in migration and winterin...,female,yes,NaN,alarm_call
4,956889,Great Tit,Estonia,59.3601,24.1552,A,0:18,8:00,2024-12-18,32000,male,male,yes,NaN,alarm_call


In [17]:
import pandas as pd

# The most reliable way: Explicit mapping for every unique string
manual_calls_mapping = {
    # --- PREDATOR (High-threat aerial/owl) ---
    "Alarm calls  near a Pygmy Owl. Exact location not disclosed.": "predator",
    "Sounds uttered upon encounter of a Pygmy Owl": "predator",
    "is sorrow on the silhouette of a hawk": "predator",
    "reaction to Sparrowhawk": "predator",
    "Possibly reacting to a raptor flying over. High-pass filtered (350Hz, 6dB). d100 + em172": "predator",
    "alarm call caused by common buzzard flying over": "predator",
    "Ejemplar alarmado al sobrevolar el águila imperial.": "predator",
    "Group actively migrating, alarm called in flight at the sight of a kestrel": "predator",
    "A pair protesting in vain as their chick is killed and eaten by an Eurasian jay (Garrulus glandarius).": "predator",
    "Great tit calls ... not filtered ... parabolic ... background noise ... little owl close by ...": "predator",
    "unbelievably fast reaction to the hawk by the Great Tit, which uttered a very high-pitched call": "predator",

    # --- INTRUDER/CAT/HUMAN/DOG (Terrestrial/Direct threats) ---
    "Scolding a cat in the cemetary": "intruder/cat/human/dog",
    "Bird perched in small tree calling in vincinity of a domestic cat on a roof close by.": "intruder/cat/human/dog",
    "Probably alarm call due to dog and my presence.": "intruder/cat/human/dog",
    "Alarm call , caused by my dog presence.": "intruder/cat/human/dog",
    "Appeared to be alarming at presence of small dog.": "intruder/cat/human/dog",
    "Common Chaffinch-like 'pink' call given by an individual that appeared to be concerned by presence of small dog.": "intruder/cat/human/dog",
    "Alarm call due to my presence near the nest box.": "intruder/cat/human/dog",
    "a pair of Great Tits approach me in warning.": "intruder/cat/human/dog",
    "a pair of Great Tits start to warn as they see me passing by": "intruder/cat/human/dog",
    "Male warning about me while feeding in a bush.": "intruder/cat/human/dog",
    "Probably alarming at me.": "intruder/cat/human/dog",
    "The bird approached and began to emit alarm calls at my presence.": "intruder/cat/human/dog",
    "calling was perceded by the great tit chasing off a intruder (male great tit)": "intruder/cat/human/dog",
    "This bird appeared to be alarming at my presence; nervously flitting around on small branches of Elm c. 1 m above my head.": "intruder/cat/human/dog",
    "P. major approached me along with a group of tits... Loud alarm calls were made": "intruder/cat/human/dog",
    "I stood about 5m in front of the nesting box and the adult bird was calling behind me": "intruder/cat/human/dog",
    "No modification. While approaching a Black Woodpecker tapping a tree, a Great Tit started alarming at me.": "intruder/cat/human/dog",
    "Two Great Tits alarming at me. Male is calling, female is churring.": "intruder/cat/human/dog",
    "Flushed male calling at me.": "intruder/cat/human/dog",
    "Responding to pishing.": "intruder/cat/human/dog",
    "Maybe alarm call, vocalization while watching us from around us.": "intruder/cat/human/dog",
    "Great tit perched in a bush was scared by the recordist's device which had fallen down": "intruder/cat/human/dog",
    "Scolding call. A rattling with an introductory see note, continuously emitted while watching me.": "intruder/cat/human/dog",

    # --- MIGRATION ---
    "usual call of female in migration and wintering phase of the year": "migration",
    "usual call in migration and wintering phase of the year": "migration",

    # --- MIMICRY ---
    "mimicry of alarm call Poecile palustris": "mimicry",
    "Bird seems to imitate calls of Poecile palustris.": "mimicry",
    "Imitación de carbonero palustre, poecile palustris.": "mimicry",
    "With some mimicry of Marsh Tit (Poecile palustris).": "mimicry",
    "Mimicry of European Crested Tit (Lophophanes cristatus) and then regular alarm call": "mimicry",
    "This is a mimicry call of Poecile palustris.": "mimicry",
    "one individual imitates a marsh tit *Poecile palustris*": "mimicry",
    "A rather *Poecile palustris* like call.": "mimicry",
    "This call sounds different from the birds I hear in France. A mimicry of Sombre Tit maybe?": "mimicry",
    "calls, including calls very much resembling Short-toed Treecreeper": "mimicry",

    # --- EATING ---
    "in a feeder": "eating",
    "Passive recording from the feeder.": "eating",
    "Excited near bird-feeding site.": "eating",
    "In my garden near the feeding-place.": "eating",
    "The bird sounded irritated by a nuthatch that stole its place at the feeder.": "eating",
    "Birds foraging in coastal forest... feeding on invertebrates": "eating",
    "con imbeccata": "eating",
    "In a bird feeder. Mountains with forest of Pinus Alepensis.": "eating",
    "In a bird feeder. Mountains wiht forest of Pinus Alepensis.": "eating",

    # --- EXCITEMENT (Agitation/Fight/Young) ---
    "excitement calls of a male": "excitement",
    "Several young birds calling excitedly": "excitement",
    "agressive": "excitement",
    "Two Great tits fighting in a pile of brushwood": "excitement",
    "3 birds seemingly fighting high up in an old oak tree": "excitement",
    "begging calls of rather young nestlings": "excitement",
    "Jeunes dans le nid - Youngs in the nest": "excitement",
    "Distress calls of an individual ringed.": "excitement",
    "worried": "excitement",
    "tit calling from the top of a birch tree, seems upset.": "excitement",
    "alarm of an adult (presence of youngs)": "excitement",
    "high-pitched 'see'-alarm calls of adults addressed at their recently fledged young": "excitement",
    "Scolding calls of a male, with very odd faint begging-type calls made by the female": "excitement",

    # --- GRP (Pairs and Flocks) ---
    "3 birds feeding in a mixed flock": "grp",
    "in a little Great Tit group.": "grp",
    "Two birds in the forest moving around from tree to tree": "grp",
    "Birds calling in mixed forest.": "grp",
    "Territorial pair": "grp",
    "Pair. The female's voice is gentle sissisi, the male's sharper.": "grp",
    "interacting pair": "grp",
    "two different calls... probably a warning call at the sudden appearance of a flyby thrush": "grp",
    "different calls by two individuals (couple?)": "grp",

    # --- URBAN ---
    "in the city, garden": "urban",
    "Suburban environment with gardens, trees, bushes and lawns": "urban",
    "Suburban area with gardens neighboring a park": "urban",
    "in residential living place": "urban",
    "in village habitat in a park": "urban",
    "in smalltown ... i moved with my rollator": "urban",
    "observed in a canopy of a young maple tree, on a sidewalk of a bypass lane": "urban",
    "Esta universidad cuenta con un campus muy rico en biodiversidad": "urban",

    # --- FOREST/TREES ---
    "Mixed forest.": "forest/trees",
    "Mixed forest, ME 66, ZOOM H5.": "forest/trees",
    "In mixed forest": "forest/trees",
    "Habitat: coniferous forest.": "forest/trees",
    "Habitat: old mixed forest.": "forest/trees",
    "Habitat: mixed forest.": "forest/trees",
    "pine - oak forest, some water": "forest/trees",
    "the oak-beech forest": "forest/trees",
    "Deciduous forest": "forest/trees",
    "Mature deciduous woodland, mainly beech trees.": "forest/trees",
    "Conifer woodland border": "forest/trees",
    "alluvial forest, cold": "forest/trees",
    "at the edge of mixed forest": "forest/trees",
    "Grabado en bosque denso de ROBLES con mucho sotobosque.": "forest/trees",
    "Grabado en bosque de ribera.": "forest/trees",
    "Bird perched in small tree": "forest/trees",
    "Bird calling while perched on a while.": "forest/trees",
    "In mixed forest by an open area": "forest/trees",
}

# Reliability function: First try exact mapping, then fall back to keywords
def apply_final_mapping(remark):
    if pd.isna(remark): return "typical"
    
    # Check manual dictionary first
    for key, value in manual_calls_mapping.items():
        if key in remark:
            return value
            
    # If not found, run technical filter
    res = str(remark).lower()
    if any(tech in res for tech in ['snr', 'hpf', 'filter', 'zoom', 'tascam', 'sd702', 'mixpre']):
        return "typical"
    
    return "typical"

df2['remarks'] = df2['remarks'].apply(apply_final_mapping)

In [19]:
df2.head()

,id,english_name,country,lat,lon,quality,length,time,date,sample_rate,remarks,sex,animal_seen,temp,label
0,1088904,Great Tit,Sweden,56.1820,14.8897,A,0:24,10:00,2026-03-15,44100,typical,male,yes,NaN,alarm_call
1,985820,Great Tit,Netherlands,53.0228,5.9170,A,0:48,08:30,2025-04-04,96000,typical,male,yes,NaN,alarm_call
2,983069,Great Tit,Netherlands,52.9638,6.1637,A,0:40,08:00,2025-03-26,48000,typical,uncertain,yes,NaN,alarm_call
3,956913,Great Tit,Estonia,59.3601,24.1552,A,0:07,8:00,2024-12-18,32000,migration,female,yes,NaN,alarm_call
4,956889,Great Tit,Estonia,59.3601,24.1552,A,0:18,8:00,2024-12-18,32000,typical,male,yes,NaN,alarm_call


In [24]:
counts = df2['remarks'].value_counts()
print(counts)

remarks
typical                   359
forest/trees               33
intruder/cat/human/dog     21
eating                     16
excitement                 12
predator                   11
mimicry                    10
grp                         8
urban                       5
migration                   2
Name: count, dtype: int64


In [27]:
df2.to_csv(meta_alarm_path)

In [28]:
df1 = pd.read_csv(meta_song_path)

In [30]:
counts = df2['sex'].value_counts()
print(counts)

sex
male            83
uncertain       55
female, male    15
female           6
Name: count, dtype: int64


In [32]:
df2['sex'] = df2['sex'].replace({
    'female, male': "female",
    'uncertain': np.nan
})
counts = df2['sex'].value_counts()
print(counts)

sex
male      83
female    21
Name: count, dtype: int64


In [33]:
counts = df2['animal_seen'].value_counts()
print(counts)

animal_seen
yes        380
no          66
unknown     31
Name: count, dtype: int64


In [34]:
df2.to_csv(meta_alarm_path)

In [36]:
counts = df1['sex'].value_counts()
print(counts)

sex
male         216
uncertain     48
Name: count, dtype: int64


In [38]:
df1['sex'] = df1['sex'].replace({
    'female, male': "female",
    'uncertain': np.nan
})
counts = df1['sex'].value_counts()
print(counts)

sex
male    216
Name: count, dtype: int64


In [39]:
df1.to_csv(meta_song_path)

In [76]:
pdf = pd.concat([df1,df2],axis=0,ignore_index= True).drop(columns=['Unnamed: 0'])
pdf.head()

,id,english_name,country,lat,lon,quality,length,time,date,sample_rate,remarks,sex,animal_seen,temp,label
0,1093386,Great Tit,China,39.9915,116.3113,A,0:35,6:09,2026-04-04,44100,typical,NaN,yes,NaN,song
1,1092854,Great Tit,Cyprus,34.7618,32.5144,A,0:31,17:00,2026-03-22,24000,typical,male,no,NaN,song
2,1092853,Great Tit,Cyprus,34.7618,32.5144,A,0:14,17:00,2026-03-22,24000,typical,male,no,NaN,song
3,1092510,Great Tit,France,46.6716,4.8614,A,0:19,10:00,2026-02-20,24000,urban,NaN,no,NaN,song
4,1092503,Great Tit,Italy,39.6118,16.7489,A,0:49,15:43,2026-03-29,44100,typical,male,yes,NaN,song


In [77]:
pdf.drop(columns=['english_name','length','temp','sample_rate','quality'], inplace=True)
pdf.head()

,id,country,lat,lon,time,date,remarks,sex,animal_seen,label
0,1093386,China,39.9915,116.3113,6:09,2026-04-04,typical,NaN,yes,song
1,1092854,Cyprus,34.7618,32.5144,17:00,2026-03-22,typical,male,no,song
2,1092853,Cyprus,34.7618,32.5144,17:00,2026-03-22,typical,male,no,song
3,1092510,France,46.6716,4.8614,10:00,2026-02-20,urban,NaN,no,song
4,1092503,Italy,39.6118,16.7489,15:43,2026-03-29,typical,male,yes,song


In [78]:
import datetime
pdf['time'] = pd.to_datetime(pdf['time'], format='%H:%M', errors='coerce')
pdf['date']= pd.to_datetime(pdf['date'],errors = 'coerce')
pdf['hour'] = pdf['time'].dt.hour
pdf['month'] = pdf['date'].dt.month
pdf['date']= pdf['date'].dt.day
pdf.head()

,id,country,lat,lon,time,date,remarks,sex,animal_seen,label,hour,month
0,1093386,China,39.9915,116.3113,1900-01-01 06:09:00,4.0,typical,NaN,yes,song,6.0,4.0
1,1092854,Cyprus,34.7618,32.5144,1900-01-01 17:00:00,22.0,typical,male,no,song,17.0,3.0
2,1092853,Cyprus,34.7618,32.5144,1900-01-01 17:00:00,22.0,typical,male,no,song,17.0,3.0
3,1092510,France,46.6716,4.8614,1900-01-01 10:00:00,20.0,urban,NaN,no,song,10.0,2.0
4,1092503,Italy,39.6118,16.7489,1900-01-01 15:43:00,29.0,typical,male,yes,song,15.0,3.0


In [80]:
pdf.drop(['time',],axis=1,inplace = True)
pdf.head()

,id,country,lat,lon,date,remarks,sex,animal_seen,label,hour,month
0,1093386,China,39.9915,116.3113,4.0,typical,NaN,yes,song,6.0,4.0
1,1092854,Cyprus,34.7618,32.5144,22.0,typical,male,no,song,17.0,3.0
2,1092853,Cyprus,34.7618,32.5144,22.0,typical,male,no,song,17.0,3.0
3,1092510,France,46.6716,4.8614,20.0,urban,NaN,no,song,10.0,2.0
4,1092503,Italy,39.6118,16.7489,29.0,typical,male,yes,song,15.0,3.0


In [81]:
pdf['n_s'] = np.where(pdf['lat']>=0,'N','S')
pdf['e_s'] = np.where(pdf['lon']>=0,'E','W')
def get_climate_zone(lat):
    abs_lat = abs(lat)
    if abs_lat < 23.5:
        return 'Tropical'
    elif abs_lat < 66.5:
        return 'Temperate'
    else:
        return 'Polar'
pdf['climate_zone'] = pdf['lat'].apply(get_climate_zone)
pdf.head()

,id,country,lat,lon,date,remarks,sex,animal_seen,label,hour,month,n_s,e_s,climate_zone
0,1093386,China,39.9915,116.3113,4.0,typical,NaN,yes,song,6.0,4.0,N,E,Temperate
1,1092854,Cyprus,34.7618,32.5144,22.0,typical,male,no,song,17.0,3.0,N,E,Temperate
2,1092853,Cyprus,34.7618,32.5144,22.0,typical,male,no,song,17.0,3.0,N,E,Temperate
3,1092510,France,46.6716,4.8614,20.0,urban,NaN,no,song,10.0,2.0,N,E,Temperate
4,1092503,Italy,39.6118,16.7489,29.0,typical,male,yes,song,15.0,3.0,N,E,Temperate


In [82]:
pdf.drop(['lat','lon'],axis=1,inplace = True)
pdf.head()

,id,country,date,remarks,sex,animal_seen,label,hour,month,n_s,e_s,climate_zone
0,1093386,China,4.0,typical,NaN,yes,song,6.0,4.0,N,E,Temperate
1,1092854,Cyprus,22.0,typical,male,no,song,17.0,3.0,N,E,Temperate
2,1092853,Cyprus,22.0,typical,male,no,song,17.0,3.0,N,E,Temperate
3,1092510,France,20.0,urban,NaN,no,song,10.0,2.0,N,E,Temperate
4,1092503,Italy,29.0,typical,male,yes,song,15.0,3.0,N,E,Temperate


In [83]:
v = pdf['n_s'].value_counts()
print(v)

n_s
N    975
S      2
Name: count, dtype: int64


In [102]:
pdf.head()

,id,country,date,remarks,sex,animal_seen,label,hour,month,n_s,e_s,climate_zone
0,1093386,China,4.0,typical,NaN,yes,song,6.0,4.0,N,E,Temperate
1,1092854,Cyprus,22.0,typical,male,no,song,17.0,3.0,N,E,Temperate
2,1092853,Cyprus,22.0,typical,male,no,song,17.0,3.0,N,E,Temperate
3,1092510,France,20.0,urban,NaN,no,song,10.0,2.0,N,E,Temperate
4,1092503,Italy,29.0,typical,male,yes,song,15.0,3.0,N,E,Temperate


In [105]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_feats = ['hour', 'month', 'date']
cat_feats = ['country', 'remarks', 'sex', 'animal_seen', 'n_s', 'e_s', 'climate_zone']

preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(),num_feats),
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown='ignore'),cat_feats)
    ],
    remainder='drop'
)
X = pdf.drop(['id','label'],axis=1)
X_preprocessed = preprocessor.fit_transform(X)

column_names = num_feats + list(preprocessor.named_transformers_['cat'].get_feature_names_out())
partial_pdf = pd.DataFrame(X_preprocessed, columns=column_names)
partial_pdf.head()

,hour,month,date,country_Albania,country_Austria,country_Azerbaijan,country_Belgium,country_China,country_Croatia,country_Cyprus,...,sex_nan,animal_seen_no,animal_seen_unknown,animal_seen_yes,n_s_N,n_s_S,e_s_E,e_s_W,climate_zone_Polar,climate_zone_Temperate
0,-1.283119,-0.090691,-1.349775,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.780298,-0.437805,0.755601,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,1.780298,-0.437805,0.755601,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,-0.169149,-0.784919,0.521670,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
4,1.223313,-0.437805,1.574358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0


In [109]:
final_pdf = pd.concat([partial_pdf,pdf[["id","label"]]],axis =1)
final_pdf.head()

,hour,month,date,country_Albania,country_Austria,country_Azerbaijan,country_Belgium,country_China,country_Croatia,country_Cyprus,...,animal_seen_unknown,animal_seen_yes,n_s_N,n_s_S,e_s_E,e_s_W,climate_zone_Polar,climate_zone_Temperate,id,label
0,-1.283119,-0.090691,-1.349775,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1093386,song
1,1.780298,-0.437805,0.755601,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1092854,song
2,1.780298,-0.437805,0.755601,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1092853,song
3,-0.169149,-0.784919,0.521670,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1092510,song
4,1.223313,-0.437805,1.574358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1092503,song


In [111]:
final_pdf = final_pdf.sort_values(by='id')
final_pdf.to_csv("final_metadata.csv")

In [ ]:
# making sure that df_index_to_npy.csv is also sorted by ids
df_index_to_npy = pd.read_csv("df_index_to_npy.csv")
df_index_to_npy['file_id_num'] = df_index_to_npy['file_id'].str.replace('.mp3', '', regex=False).astype(int)

df_index_to_npy = df_index_to_npy.sort_values(by='file_id_num').drop(columns='file_id_num')
df_index_to_npy.head()
df_index_to_npy.to_csv('df_index_to_npy.csv',index=False)

,file_id,label,npy_path,file_id_num
0,22726.mp3,alarm_call,C:\Users\HishamMo\Desktop\999_cs\animal-stress...,22726
1,28218.mp3,song,C:\Users\HishamMo\Desktop\999_cs\animal-stress...,28218
2,32820.mp3,song,C:\Users\HishamMo\Desktop\999_cs\animal-stress...,32820
3,42775.mp3,song,C:\Users\HishamMo\Desktop\999_cs\animal-stress...,42775
4,43760.mp3,song,C:\Users\HishamMo\Desktop\999_cs\animal-stress...,43760


In [123]:
for i in final_pdf["id"]:
    if i not in df_index_to_npy['file_id_num'].values:
        print(i)

1083438
